In [ ]:
import io
import os
import glob
from datetime import datetime
import pandas as pd
import pyarrow.parquet as pq
from termcolor import colored
from google.colab import drive
drive.mount('/content/gdrive')
#%%


In [ ]:
data_dir_path      = '/content/gdrive/MyDrive/BankHoldingCompanyData/bhc_tex_files/'
script_dir_path    = '/content/gdrive/MyDrive/BankHoldingCompanyData/'
bhc_parquet_dir    = '/content/gdrive/MyDrive/BankHoldingCompanyData/bhc_parquet_files/'
os.chdir(script_dir_path)

In [ ]:
def check_file_exists(longfname):
    return os.path.isfile(longfname)


def makelable_dict(df):
    lijst_df_rssd = list(set(x for x in df.columns if 'RSSD' in x))
    lijst_df_bhck = list(set(x[-4:] for x in df.columns if 'year' not in x and 'qid' not in x and 'RSSD' not in x))

    dfd = pd.read_csv("MDRM_CSV.csv", encoding="ISO-8859-1", low_memory=False, parse_dates=False, skiprows=[0])
    dfd = dfd.loc[dfd['Reporting Form'] == "FR Y-9C", ['Mnemonic', 'Item Code', 'Item Name']]
    dfd = dfd.rename(columns={'Item Code': 'item', 'Item Name': 'label', 'Mnemonic': 'mnem'}).drop_duplicates(subset=['item'])
    dfd = dfd.assign(long_mnem=dfd.mnem + dfd.item)

    df_rssd  = pd.DataFrame(index=lijst_df_rssd).join(dfd.set_index('long_mnem')['label']).dropna()
    df_FRY9C = pd.DataFrame(index=lijst_df_bhck).join(dfd.set_index('item')['label']).dropna()

    df_lables = pd.DataFrame(
        [x for x in df.columns if 'year' not in x and 'qid' not in x and 'RSSD' not in x],
        columns=['long_key']
    )
    df_lables['item'] = df_lables['long_key'].str[-4:]
    df_lables = df_lables.set_index('item').join(df_FRY9C).reset_index().dropna()
    df_lables = df_lables[['long_key', 'label']].set_index('long_key')
    df_lables.index.names = ['item']

    df_lables = pd.concat([df_lables, df_rssd])
    df_lables['label'] = df_lables['label'].str[:50]
    return df_lables.to_dict()['label']


def concat_pieces(pieces, fname, parquetfile, statafile, label):
    if not pieces:
        print(f"{label} is empty")
        return None
    df = pd.concat(pieces, ignore_index=True, sort=False)
    df = order(df, ['RSSD9001', 'RSSD9999', 'year', 'qid'])
    print('\nWriting to csv.gz')
    df.to_csv(fname + '.gz', index=False, sep="^", compression='gzip')
    print('\nWriting to parquet')
    df.to_parquet(parquetfile)
    labels_csv = parquetfile.replace('.parquet', '_labels.csv')
    label_dict = makelable_dict(df)
    pd.Series(label_dict, name='label').rename_axis('item').to_csv(labels_csv)
    print(f'\nWriting labels to {labels_csv}')
    if statafile != "0":
        print('\nWriting to Stata')
        df.to_stata(statafile, version=119, variable_labels=label_dict)
    print(f'\n{label} has a count of {df["qid"].count()}.')
    return df


def order(frame, var):
    varlist = [w for w in frame.columns if w not in var]
    return frame[var + varlist]


def delfile(file_to_delete):
    if file_to_delete != "0":
        try:
            os.remove(file_to_delete)
        except OSError:
            pass


def writefilesorted(lst, pad, file_to_write):
    with open(pad + file_to_write, mode='wt') as myfile:
        myfile.write('\n'.join(sorted(lst)))


def qtr(x):
    return {'0331': 1, '0630': 2, '0930': 3, '1231': 4}[x]


def read_vars(pad, fname, up):
    columns = pd.read_csv(pad + fname, header=None).apply(lambda x: x.str.strip())[0].tolist()
    if up == "U":
        columns = [x.upper() for x in columns]
    return columns


def read_file_content(fname):
    """Read file with ISO-8859-1 encoding. For bhcf0303.txt, filter the known
    bad line (bank 1115406, which contains an embedded EOF character) to avoid
    corrupting the source file."""
    with open(fname, encoding='ISO-8859-1') as f:
        lines = f.readlines()
    if "bhcf0303.txt" in fname:
        filtered = [line for line in lines if line.split('^')[0] != "1115406"]
        skipped = len(lines) - len(filtered)
        if skipped:
            print(colored(f"\nSkipped {skipped} bad line(s) in {os.path.basename(fname)}.\n", 'red'))
        return ''.join(filtered)
    return ''.join(lines)


def load_bhc_file(fname, parquet_cache_dir):
    """Load a single BHC file, using a parquet cache when available.

    On first encounter the txt is read, parsed, and saved as parquet so future
    runs skip the slower text parsing entirely.  bhcf0303.txt is handled
    transparently via read_file_content().

    Returns a DataFrame with uppercased column names and year/qid added.
    """
    os.makedirs(parquet_cache_dir, exist_ok=True)
    base       = os.path.splitext(os.path.basename(fname))[0]
    cache_path = os.path.join(parquet_cache_dir, base + '.parquet')

    if os.path.isfile(cache_path):
        print(colored(" [from cache]", 'cyan'), end='')
        return pd.read_parquet(cache_path)

    # --- parse the txt file ---
    content = read_file_content(fname)
    buf     = io.StringIO(content)

    word = buf.readline().upper().split('^')
    if word[0] == "RSSD9001":
        pass  # idx not needed for rowcount scan
    elif word[1] == "RSSD9001":
        pass
    else:
        raise ValueError(f"Cannot find RSSD9001 in header of {fname}")

    rowcount = sum(1 for line in buf if line.split('^')[0] == "--------")

    buf = io.StringIO(content)
    if rowcount:
        dfr = pd.read_csv(buf, sep='^', parse_dates=False, skiprows=[rowcount], engine="python", on_bad_lines='skip')
    else:
        dfr = pd.read_csv(buf, sep='^', parse_dates=False, engine="python", on_bad_lines='skip')

    dfr.columns = [c.upper() for c in dfr.columns]
    dfr['year'] = int(str(dfr["RSSD9999"].iloc[0])[0:4])
    dfr['qid']  = qtr(str(dfr["RSSD9999"].iloc[0])[4:9])

    dfr.to_parquet(cache_path)
    print(colored(f" [cached → {os.path.basename(cache_path)}]", 'cyan'), end='')
    return dfr

In [ ]:
def main(pad, data_dir_path, parquet_cache_dir, path, bank_out_file, var_out_file, panelfile, parquetfile, vars_in_file, filesfile, statafile):
    banks = []
    variables = []
    all_pieces = []
    filecount = 0

    tgt_list = read_vars(pad, vars_in_file, "U")

    if check_file_exists(pad + filesfile):
        lyst = read_vars(pad, filesfile, "L")
        add_pad = True
    else:
        lyst = sorted(glob.glob(path))
        add_pad = False

    tot_filecount = len(lyst)
    print(f'\nTotal number of files submitted: {tot_filecount}.\n')

    for naam in lyst:
        fname = (data_dir_path + naam) if add_pad else naam
        print(f"\nFile: {fname}", end='')

        if check_file_exists(fname):
            dfr = load_bhc_file(fname, parquet_cache_dir)
            filecount += 1

            for bid in dfr['RSSD9001'].astype(str):
                if bid not in banks:
                    banks.append(bid)

            for item in dfr.columns:
                if item not in variables:
                    variables.append(item)

            var_list = list(dfr.columns)
            print(f' Cell 1: {var_list[0]:>10}. Cell 2: {var_list[1]:>10}. Cell 3: {var_list[2]:>10}.', end='')
            print(' RSSD9999: ', end='')
            print(colored(dfr["RSSD9999"].iloc[0], 'red'), end='')
            print(f' Year: {dfr["year"].iloc[0]:5}. Qtr: {dfr["qid"].iloc[0]:2}. Count {dfr["RSSD9001"].count():4}. Vars: {len(var_list):5}', end='')
            print(f' Banks: {len(banks):5}.', end='')
            print(f' Variables: {len(variables):5}.', end='')
            print(f' Filecount: {filecount:4} of {tot_filecount:4}.\n', end='')

            tmp_list = ['RSSD9001', 'RSSD9999', 'year', 'qid']
            for item in tgt_list:
                if item in var_list:
                    if item not in tmp_list:
                        tmp_list.append(item)
                        print(colored(item + " ", 'green'), end='')
                else:
                    print(colored(item + " ", 'red'), end='')
            dfr = dfr[tmp_list]
            all_pieces.append(dfr)
        else:
            print(colored(f"\nThis BHC file is missing: {fname}\nI will skip.\n", 'red'))
            filecount += 1

    df = concat_pieces(all_pieces, panelfile, parquetfile, statafile, 'Panel')

    print("\nTotals")
    print(f'Banks:     {len(banks)}.')
    print(f'Variables: {len(variables)}.')
    writefilesorted(banks,     pad, bank_out_file)
    writefilesorted(variables, pad, var_out_file)

    # Observation counts per quarter
    counts = df.groupby("RSSD9999")["RSSD9999"].count().rename("count")
    print("\nObservations per quarter:")
    print(counts.to_string())

    # Save summary to markdown
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    md_file = parquetfile.replace('.parquet', f'_{ts}.md')
    with open(md_file, 'wt') as f:
        f.write(f"# Run summary — {ts}\n\n")
        f.write(f"## Totals\n\n")
        f.write(f"- Banks: {len(banks)}\n")
        f.write(f"- Variables: {len(variables)}\n\n")
        f.write(f"## Observations per quarter\n\n")
        f.write("| RSSD9999 | count |\n|----------|-------|\n")
        for rssd, n in counts.items():
            f.write(f"| {rssd} | {n} |\n")
        total = counts.sum()
        f.write(f"| **Total** | **{total}** |\n")
    print(f"\nSummary written to {md_file}")

    print("\nDone!!!!\n")
    return df


In [ ]:
args = {
    "bank_out_file": 'banks.csv',
    "var_out_file":  'variables.csv',
    "panelfile":     'panelfile.csv',
    "parquetfile":   'panelfile.parquet',
    "vars_in_file":  'bhc_vars.csv',
    "filesfile":     'lyst3.csv',
    "statafile":     '0',
}

pad  = script_dir_path
path = data_dir_path + "bhcf*.txt"

print("\nInputs:")
for key, value in args.items():
    print(f"  {key}: {value}")
print(f"  pad: {pad}")
print(f"  data_dir_path: {data_dir_path}")
print(f"  bhc_parquet_dir: {bhc_parquet_dir}")
print(f"  path: {path}")
print()

df = main(pad, data_dir_path, bhc_parquet_dir, path, **args)